In [ ]:
import os, subprocess, sys

REPO_URL = 'https://github.com/NguyenAn05/pocket-r1.git'
REPO_DIR = '/tmp/pocket-r1'
LORA_DIR = '/kaggle/input/pocket-r1-exp1b-lora'
MERGED_DIR = '/tmp/exp1b_sft_fim_merged'
OUT_REPORT_DIR = '/kaggle/working/exp1b_sft_fim'
os.makedirs(OUT_REPORT_DIR, exist_ok=True)

print(subprocess.check_output(['nvidia-smi', '--query-gpu=index,name,memory.total', '--format=csv'], text=True))
print('Mounted LoRA files:', os.listdir(LORA_DIR))

if os.path.exists(REPO_DIR):
    subprocess.check_call(['rm', '-rf', REPO_DIR])
subprocess.check_call(['git', 'clone', '--depth', '1', REPO_URL, REPO_DIR])

# Remove incompatible preinstalled torchao 0.10.0 so peft merge works cleanly
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao'], check=False)
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-e', REPO_DIR])
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'peft', 'vllm'])
print('Setup complete!')


In [ ]:
import sys
sys.path.insert(0, f'{REPO_DIR}/src')
from pocket_r1.training import merge_lora_adapter

merge_lora_adapter('Qwen/Qwen3-4B', LORA_DIR, MERGED_DIR)
print('Merged Exp 1b FP16 model ready at:', MERGED_DIR)


In [ ]:
import subprocess, time, urllib.request, sys, os

LOG_PATH = '/tmp/vllm_exp1b.log'
log_file = open(LOG_PATH, 'w')

vllm_cmd = [
    sys.executable, '-m', 'vllm.entrypoints.openai.api_server',
    '--model', MERGED_DIR,
    '--served-model-name', 'Qwen/Qwen3-4B-exp1b-sft-fim',
    '--dtype', 'half',
    '--tensor-parallel-size', '2',
    '--max-model-len', '2048',
    '--gpu-memory-utilization', '0.85',
    '--enforce-eager',
    '--host', '127.0.0.1',
    '--port', '8000',
]
print('Starting vLLM for Exp 1b:', ' '.join(vllm_cmd))
vllm_proc = subprocess.Popen(vllm_cmd, stdout=log_file, stderr=subprocess.STDOUT)

ready = False
for sec in range(420):
    if vllm_proc.poll() is not None:
        log_file.flush()
        with open(LOG_PATH) as f:
            print('vLLM exited early! Last logs:\n', f.read()[-4000:])
        raise RuntimeError(f'vLLM crashed with code {vllm_proc.returncode}')
    try:
        with urllib.request.urlopen('http://127.0.0.1:8000/v1/models', timeout=2) as r:
            if r.status == 200:
                print(f'vLLM ready in {sec}s:', r.read().decode())
                ready = True
                break
    except Exception:
        time.sleep(1)

if not ready:
    vllm_proc.terminate()
    raise TimeoutError('vLLM did not become ready within 420s')

try:
    eval_cmd = [
        sys.executable, f'{REPO_DIR}/scripts/run_exp0_baseline.py',
        '--config', f'{REPO_DIR}/configs/eval/exp0_baseline.yaml',
        '--backend', 'openai_compatible',
        '--model', 'Qwen/Qwen3-4B-exp1b-sft-fim',
        '--base-url', 'http://127.0.0.1:8000/v1',
        '--workers', '24',
        '--no-ev',
        '--experiment-name', 'exp1b_sft_fim',
        '--output-dir', OUT_REPORT_DIR,
    ]
    print('Evaluating Exp 1b on 2000 Held-out hands:', ' '.join(eval_cmd))
    subprocess.check_call(eval_cmd, cwd=REPO_DIR)
finally:
    vllm_proc.terminate()
    try:
        vllm_proc.wait(timeout=20)
    except Exception:
        vllm_proc.kill()
    log_file.close()
    print('Exp 1b Eval complete! Outputs:', os.listdir(OUT_REPORT_DIR))
